# 01_translate_svg

In [1]:
# Set languages and LLM model

source_language = 'English' # for LLM prompting
target_language = 'Traditional Chinese' # for LLM prompting

gemini_model_name = 'gemini-3.1-pro-preview' # Primary translator


In [2]:
# Load API key
# API key must be in a .env file in working directory (bst)

import os
from dotenv import load_dotenv
from google import genai

# Load .env once
load_dotenv()

# Force Gemini key usage (paid access)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in environment")

# OPTIONAL: prevent accidental fallback
# This avoids the SDK silently choosing GOOGLE_API_KEY
os.environ.pop("GOOGLE_API_KEY", None)

# Create a single reusable client
genai_client = genai.Client(api_key=GEMINI_API_KEY)

# print(f"Gemini client initialized with key: {GEMINI_API_KEY[:4]}…{GEMINI_API_KEY[-4:]}")


## System message
**CAUTION:** Do not customize the following cell

In [3]:
# do not customize this cell

def build_svg_units_system_instruction(
    source_language: str,
    target_language: str,
    domain_context: str | None = None,
) -> str:
    custom_block = ""
    if domain_context:
        custom_block = f"""
   - Domain context:
     {domain_context}
""".rstrip()

    return f"""
You are a professional translator.

You will receive ONE JSON object with this structure:

{{
  "units": [
    {{
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in {source_language}>"
    }},
    ...
  ]
}}

Translate each `source_text` from {source_language} into {target_language}.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

You must respond with ONE valid JSON object of the form:

{{
  "units": [
    {{
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in {target_language}>"
    }},
    ...
  ]
}}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "units" must have the same number of entries as the input, in the same order.
   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".
   - Do not include "group_stack" or "source_text" in the output.
   - Output must be valid JSON: no trailing commas, no comments, no code fences, no extra text.
   - Each "unit_key" must appear exactly once in the output. Do not duplicate, omit, or invent unit_key values.
   - The i-th output unit must correspond to the i-th input unit (same order).

2) Text handling:
   - Preserve numbers, punctuation, and abbreviations when meaningful.
   - Keep proper nouns unchanged unless a standard {target_language} form exists.
   - If `source_text` is already primarily in {target_language}, copy it unchanged.
   - Use wording and idiom appropriate to the source context indicated by `group_stack`.{custom_block}

3) Numbers and mixed text:
   - Preserve all numbers exactly as they appear in the source_text.
   - When source_text contains both words and numbers, translate only the words and keep all numbers unchanged and in the same relative position.
   - Do not convert numbers to words, do not reformat them, and do not add or remove numeric content.

4) Brevity:
   - These are short labels for a graphic. Prefer concise translations that fit similar space.

Your entire response must be ONLY the JSON object described above.
""".strip()

### Domain context (optional)
- Update `domain_context` below if this SVG set needs brief project-specific translation guidance.
- Keep it short and focused on terminology or interpretive context specific to this project.

In [4]:
primary_system_message = build_svg_units_system_instruction(
    source_language=source_language,
    target_language=target_language,
    domain_context=(
        "This material contains Biblical and historical labels. "
        "Use terminology natural to a Protestant and Evangelical Christian context. "
        "Preserve established Biblical names and standard historical/Biblical terms in the target language."
    ),
)

In [5]:
print(primary_system_message)

You are a professional translator.

You will receive ONE JSON object with this structure:

{
  "units": [
    {
      "unit_key": "<string>",
      "group_stack": "<string describing the SVG layer/group context>",
      "source_text": "<text in English>"
    },
    ...
  ]
}

Translate each `source_text` from English into Traditional Chinese.
The `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.

You must respond with ONE valid JSON object of the form:

{
  "units": [
    {
      "unit_key": "<same unit_key as input>",
      "translated_text": "<translated text in Traditional Chinese>"
    },
    ...
  ]
}

REQUIREMENTS

1) JSON contract:
   - Return exactly one top-level JSON object with exactly one key: "units".
   - "units" must have the same number of entries as the input, in the same order.
   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".
   - Do not include "group_stack" or "source_text" in t

In [6]:
print(repr(primary_system_message))

'You are a professional translator.\n\nYou will receive ONE JSON object with this structure:\n\n{\n  "units": [\n    {\n      "unit_key": "<string>",\n      "group_stack": "<string describing the SVG layer/group context>",\n      "source_text": "<text in English>"\n    },\n    ...\n  ]\n}\n\nTranslate each `source_text` from English into Traditional Chinese.\nThe `group_stack` is context to help you choose appropriate wording, capitalization, and abbreviations.\n\nYou must respond with ONE valid JSON object of the form:\n\n{\n  "units": [\n    {\n      "unit_key": "<same unit_key as input>",\n      "translated_text": "<translated text in Traditional Chinese>"\n    },\n    ...\n  ]\n}\n\nREQUIREMENTS\n\n1) JSON contract:\n   - Return exactly one top-level JSON object with exactly one key: "units".\n   - "units" must have the same number of entries as the input, in the same order.\n   - Each output entry must contain exactly two keys: "unit_key" and "translated_text".\n   - Do not includ

In [7]:
print("Domain context present:", "Protestant and Evangelical Christian context" in primary_system_message)

Domain context present: True


In [8]:
print("Using system message with length:", len(primary_system_message))

Using system message with length: 2422


In [10]:
def gemini_generate(payload_json_str: str, timeout: int = 120):
    return genai_client.models.generate_content(
        model=gemini_model_name,
        contents=payload_json_str,
        system_instruction=primary_system_message,
        request_options={"timeout": timeout},
    )

print("Gemini client ready:", genai_client)


Gemini client ready: <google.genai.client.Client object at 0x00000250C8D40710>


## Import JSON payload

In [11]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"
assert SVG_OUTPUT_DIR.exists(), f"Missing folder: {SVG_OUTPUT_DIR}"
assert JSON_DIR.exists(), f"Missing folder: {JSON_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: bst
SVG_SOURCE_DIR: bst\svg_source_files
JSON_DIR: bst\json_files
SVG_OUTPUT_DIR: bst\svg_output_files


In [12]:
import json
from pathlib import Path
import pandas as pd

# --- locate and load the translation-units JSON produced in the previous notebook ---

units_path = JSON_DIR / "translation_units.json"
assert units_path.exists(), f"Missing file: {units_path}"

units = json.loads(units_path.read_text(encoding="utf-8"))
assert isinstance(units, list), "Expected a JSON array of unit records."

df_units = pd.DataFrame(units)
required_cols = {"unit_key", "unit_type", "source_file", "group_stack", "element_path", "source_text"}
missing = required_cols - set(df_units.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

df_units["source_text"] = df_units["source_text"].fillna("").astype(str)
df_units = df_units[df_units["source_text"].str.strip().ne("")].copy()
df_units.reset_index(drop=True, inplace=True)

print("Loaded units:", len(df_units))
print("Files:", df_units["source_file"].nunique())
df_units.head(10)


Loaded units: 199
Files: 2


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
0,dff2455410d362f483bc04508cbcb5753bca52e3,tspan,BibleStructureTimelineNT_2024.svg,Headline,svg/g#Headline/text[1],None,None,0,The Resurrection of Jesus is the Best Attested...,
1,28f9e896771634fe5b28da1b42499b9af0f75317,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[1],None,None,0,History,
2,f68d6449d4fb305e4ee8fbc2c63e27b3c8804cbf,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[2],None,None,0,John the Apostle,
3,a4653f3314eea5bd559a4ee5a5d4fe270c468cde,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[3],None,None,0,Matthew the Apostle,
4,868584564de32b98e7ea2cdfbf70b0fff67c3f03,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[4],None,None,0,John Mark,
5,bb0c7e00990a6f81b12f907151f44957605ce468,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[5],None,None,0,"Luke the Physician, Companion of Paul",
6,32425388bc206477680f01e421b5f721020989c8,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[6],None,None,0,"Luke the Physician, Companion of Paul",
7,7aef81e2f7aec81230d0a96a083f7441fb6d6ac3,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Acts_Box,svg/g#NT_History/g#Acts_Box/text[1],None,None,1,Acts,
8,b14fd7d7025e63af23543b038ac1a4fbd28dacb0,tspan,BibleStructureTimelineNT_2024.svg,NT_History / John_Box,svg/g#NT_History/g#John_Box/text[1],None,None,1,John,
9,d279e3b729c3b4b8eef8d006af038782951c6807,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Luke_Box,svg/g#NT_History/g#Luke_Box/text[1],None,None,1,Luke,


## Prepare units
### Packet sizing
- These settings control how translation units are grouped into API requests.
- In most cases, leave these values unchanged.  
Only reduce them if requests are failing, timing out, or returning malformed/incomplete JSON.  
Increase them only if you are intentionally testing larger batches and understand the risk of oversized requests or harder-to-debug failures.
- A small final packet is normal and expected with this packing method.

In [43]:
# --- packetize units into request-sized chunks (simple char-based packing) ---
# Adjust these if you want larger/smaller packets; char-based is predictable across models.

# MAX_CHARS_PER_PACKET = 12000   # conservative for prompts + JSON overhead
# MAX_UNITS_PER_PACKET = 200     # hard cap to keep responses manageable

MAX_CHARS_PER_PACKET = 6000
MAX_UNITS_PER_PACKET = 100

def packetize_units(df: pd.DataFrame,
                    max_chars: int = MAX_CHARS_PER_PACKET,
                    max_units: int = MAX_UNITS_PER_PACKET):
    packets = []
    current = []
    current_chars = 0

    # Stable ordering helps reproducibility and diffing
    df_sorted = df.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"], na_position="last")

    for _, r in df_sorted.iterrows():
        rec = {
            "unit_key": r["unit_key"],
            "group_stack": r["group_stack"],
            "source_text": r["source_text"],
        }
        # Estimate size as JSON string length
        rec_chars = len(json.dumps(rec, ensure_ascii=False))
        if rec_chars > max_chars:
            raise ValueError(f"Single unit exceeds max_chars ({rec_chars} > {max_chars}): {rec['unit_key']}")

        # Start new packet if needed
        if current and ((current_chars + rec_chars) > max_chars or (len(current) >= max_units)):
            packets.append(current)
            current = []
            current_chars = 0

        current.append(rec)
        current_chars += rec_chars

    if current:
        packets.append(current)

    return packets

packets = packetize_units(df_units)

print("Packets:", len(packets))
print("Units per packet (first 10):", [len(p) for p in packets[:10]])
print("Approx chars per packet (first 3):", [sum(len(json.dumps(u, ensure_ascii=False)) for u in p) for p in packets[:3]])


Packets: 5
Units per packet (first 10): [48, 49, 48, 51, 3]
Approx chars per packet (first 3): [5982, 5903, 5996]


In [44]:
# --- build request payloads ready to send to Gemini (each payload is a single JSON object) ---
request_payloads = [{"units": p} for p in packets]

# quick peek at one payload
print("Example payload keys:", request_payloads[0].keys())
print("Example payload unit count:", len(request_payloads[0]["units"]))
print(json.dumps(request_payloads[0]["units"][0], ensure_ascii=False, indent=2))


Example payload keys: dict_keys(['units'])
Example payload unit count: 48
{
  "unit_key": "dff2455410d362f483bc04508cbcb5753bca52e3",
  "group_stack": "Headline",
  "source_text": "The Resurrection of Jesus is the Best Attested Event in History"
}


In [45]:
# inspect an entry

request_payloads[4]["units"][1]

{'unit_key': '5d028c27658db9e4be67aecb81573399149038cb',
 'group_stack': 'OT_Timeline',
 'source_text': 'Law'}

### Inspect payload
- Pick a packet with a small number of entries using `packet_index`

In [46]:
# inspect payload
# pick one with a small number of entries

packet_index = 4

request_payload = {"units": packets[packet_index]}

request_debug = {
    "model": gemini_model_name,
    "system_instruction": primary_system_message,
    "contents": request_payload,   # dict, not string
}

from pprint import pprint
pprint(request_debug)

{'contents': {'units': [{'group_stack': 'OT_Timeline',
                         'source_text': 'Freedom',
                         'unit_key': '9899ae9b8033df1f175581aeba0b7718f509c3e3'},
                        {'group_stack': 'OT_Timeline',
                         'source_text': 'Law',
                         'unit_key': '5d028c27658db9e4be67aecb81573399149038cb'},
                        {'group_stack': 'OT_Timeline / g',
                         'source_text': '? - 2000bc',
                         'unit_key': 'ce4cf7aafc17ed69a8537a572a1ddd2a5aff3370'}]},
 'model': 'gemini-3.1-pro-preview',
 'system_instruction': 'You are a professional translator.\n'
                       '\n'
                       'You will receive ONE JSON object with this structure:\n'
                       '\n'
                       '{\n'
                       '  "units": [\n'
                       '    {\n'
                       '      "unit_key": "<string>",\n'
                       '      "group_

### Send one test packet

In [47]:
import json
import time
from google.genai import types

print("Preparing request...")
print("Model:", gemini_model_name)
print("Units in payload:", len(request_payload["units"]))
print("Sending request to Gemini...")

t0 = time.time()

response = genai_client.models.generate_content(
    model=gemini_model_name,
    contents=json.dumps(request_payload, ensure_ascii=False),
    config=types.GenerateContentConfig(
        system_instruction=primary_system_message,
    ),
)

elapsed = time.time() - t0
print(f"Response received in {elapsed:.2f} seconds")

print("Preview of response text:")
print(response.text[:2000])

Preparing request...
Model: gemini-3.1-pro-preview
Units in payload: 3
Sending request to Gemini...
Response received in 28.28 seconds
Preview of response text:
```json
{
  "units": [
    {
      "unit_key": "9899ae9b8033df1f175581aeba0b7718f509c3e3",
      "translated_text": "自由"
    },
    {
      "unit_key": "5d028c27658db9e4be67aecb81573399149038cb",
      "translated_text": "律法"
    },
    {
      "unit_key": "ce4cf7aafc17ed69a8537a572a1ddd2a5aff3370",
      "translated_text": "? - 主前2000"
    }
  ]
}
```


In [48]:
# strip code fences
# One-off cleanup + parse for Gemini responses wrapped in ```json ... ```
import json

raw = response.text

# Strip leading/trailing whitespace first
s = raw.strip()

# If wrapped in fenced code block, remove the fences
if s.startswith("```"):
    lines = s.splitlines()
    # drop first line: ``` or ```json
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    # drop last line: ```
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    s = "\n".join(lines).strip()

# Now parse JSON
parsed = json.loads(s)

# Basic validation
assert "units" in parsed, "Missing top-level key 'units'"
assert len(parsed["units"]) == len(request_payload["units"]), (
    f"Unit count mismatch: got {len(parsed['units'])}, expected {len(request_payload['units'])}"
)

print("Parsed OK | units:", len(parsed["units"]))
print("First 2:", parsed["units"][:2])

Parsed OK | units: 3
First 2: [{'unit_key': '9899ae9b8033df1f175581aeba0b7718f509c3e3', 'translated_text': '自由'}, {'unit_key': '5d028c27658db9e4be67aecb81573399149038cb', 'translated_text': '律法'}]


In [49]:
# verify packets (input vs model output)
def validate_units(result: dict, input_payload: dict):
    out_units = result.get("units", [])
    in_units = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert len(out_keys) == len(in_keys), f"Count mismatch: out={len(out_keys)} in={len(in_keys)}"
    assert out_keys == in_keys, "unit_key sequence mismatch (duplicates, missing, or re-ordered keys)."

# use the variables you actually have right now
validate_units(parsed, request_payload)
print("Validation OK")


Validation OK


### Run all packets
#### Load helper functions

In [50]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [51]:
from collections import Counter

def debug_unit_key_mismatch(result: dict, input_payload: dict, n_show: int = 20):
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    dup_out = [k for k, c in out_counts.items() if c > 1]
    dup_in  = [k for k, c in in_counts.items() if c > 1]

    print("Input units:", len(in_keys), "| Output units:", len(out_keys))
    print("Missing keys (in input but not output):", len(missing))
    print("Extra keys (in output but not input):", len(extra))
    print("Duplicates in output:", len(dup_out))
    print("Duplicates in input:", len(dup_in))

    if missing:
        print("\nFirst missing keys:")
        for k in missing[:n_show]:
            print("  ", k)

    if extra:
        print("\nFirst extra keys:")
        for k in extra[:n_show]:
            print("  ", k)

    if dup_out:
        print("\nFirst duplicated output keys:")
        for k in dup_out[:n_show]:
            print("  ", k, "count=", out_counts[k])

    # show first index where order differs (if lengths match)
    if len(out_keys) == len(in_keys):
        for i, (ok, ik) in enumerate(zip(out_keys, in_keys)):
            if ok != ik:
                print(f"\nFirst order mismatch at index {i}:")
                print("  expected:", ik)
                print("  got     :", ok)
                break


In [52]:
import json
import time
import re
import pandas as pd
import httpx
from google.genai import types, errors as genai_errors

def extract_json_object(text: str) -> dict:
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t)
    return json.loads(t)

def generate_with_retry(client, model_name: str, system_prompt: str, payload_obj: dict,
                        retries: int = 4, backoff_s: float = 5.0):
    payload_str = json.dumps(payload_obj, ensure_ascii=False)
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(
                model=model_name,
                contents=payload_str,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                ),
            )

        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            last_err = e
            status = getattr(e, "status_code", None)
            msg = str(e)
            transient = (
                status in {429, 500, 503, 504}
                or "RESOURCE_EXHAUSTED" in msg
                or "UNAVAILABLE" in msg
                or "DEADLINE" in msg
            )
            if (attempt == retries) or (not transient):
                raise
            print(f"retry {attempt}/{retries} after API error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

        except (httpx.RemoteProtocolError, httpx.ReadTimeout, httpx.ConnectTimeout,
                httpx.ConnectError, httpx.NetworkError, httpx.TransportError) as e:
            last_err = e
            if attempt == retries:
                raise
            print(f"retry {attempt}/{retries} after transport error: {type(e).__name__}: {e}")
            time.sleep(backoff_s * attempt)

    raise last_err

In [58]:
from collections import Counter

def validate_units_keys_only(result: dict, input_payload: dict):
    """
    Validates that output contains exactly the same unit_keys as input,
    with the same counts, regardless of ordering.
    """
    out_units = result.get("units", [])
    in_units  = input_payload.get("units", [])

    out_keys = [u.get("unit_key") for u in out_units]
    in_keys  = [u.get("unit_key") for u in in_units]

    assert None not in out_keys, "Output contains a unit missing unit_key."
    assert None not in in_keys,  "Input contains a unit missing unit_key."

    out_counts = Counter(out_keys)
    in_counts  = Counter(in_keys)

    missing = [k for k in in_counts if out_counts[k] == 0]
    extra   = [k for k in out_counts if in_counts[k] == 0]
    wrong_count = [k for k in in_counts if out_counts[k] != in_counts[k]]

    assert not missing, f"Missing unit_keys in output (first 5): {missing[:5]}"
    assert not extra,   f"Extra unit_keys in output (first 5): {extra[:5]}"
    assert not wrong_count, f"Count mismatch for some keys (first 5): {wrong_count[:5]}"

def reorder_units_to_input_order(result: dict, input_payload: dict) -> dict:
    """
    Reorders result["units"] to match the order of input_payload["units"].
    Assumes validate_units_keys_only has passed.
    """
    out_units = result.get("units", [])
    in_units  = input_payload["units"]

    buckets = {}
    for u in out_units:
        k = u["unit_key"]
        buckets.setdefault(k, []).append(u)

    ordered = []
    for u_in in in_units:
        k = u_in["unit_key"]
        ordered.append(buckets[k].pop(0))

    leftovers = sum(len(v) for v in buckets.values())
    assert leftovers == 0, "Internal reorder error: leftover output units exist."

    return {"units": ordered}

In [53]:
# import json
# import time
# import re
# import pandas as pd
# from google.genai import types, errors as genai_errors

# def extract_json_object(text: str) -> dict:
#     t = (text or "").strip()
#     t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
#     t = re.sub(r"\s*```$", "", t)
#     return json.loads(t)

# def generate_with_retry(client, model_name: str, system_prompt: str, payload_obj: dict,
#                         retries: int = 3, backoff_s: float = 5.0):
#     payload_str = json.dumps(payload_obj, ensure_ascii=False)
#     last_err = None

#     for attempt in range(1, retries + 1):
#         try:
#             return client.models.generate_content(
#                 model=model_name,
#                 contents=payload_str,
#                 config=types.GenerateContentConfig(
#                     system_instruction=system_prompt,
#                 ),
#             )
#         except (genai_errors.ClientError, genai_errors.ServerError) as e:
#             last_err = e
#             status = getattr(e, "status_code", None)
#             msg = str(e)
#             transient = (
#                 status in {429, 500, 503, 504}
#                 or "RESOURCE_EXHAUSTED" in msg
#                 or "UNAVAILABLE" in msg
#                 or "DEADLINE" in msg
#             )
#             if (attempt == retries) or (not transient):
#                 raise
#             time.sleep(backoff_s * attempt)

#     raise last_err

In [54]:
# # --- run all packets ---
# all_out = []
# for i, payload in enumerate(request_payloads):
#     print(f"Packet {i+1}/{len(request_payloads)} ...", end=" ", flush=True)

#     resp = generate_with_retry(
#         client=genai_client,
#         model_name=gemini_model_name,
#         system_prompt=primary_system_message,
#         payload_obj=payload,
#         retries=3,
#         backoff_s=5,
#     )

#     result = extract_json_object(resp.text)

#     validate_units_keys_only(result, payload)
#     result = reorder_units_to_input_order(result, payload)

#     all_out.extend(result["units"])

#     print("OK")

# df_trans = pd.DataFrame(all_out)
# print("Total translated units:", len(df_trans))
# df_trans.head(10)

In [59]:
# --- run all packets ---
all_out = []

for i, payload in enumerate(request_payloads):
    print(f"Packet {i+1}/{len(request_payloads)} ...", end=" ", flush=True)

    try:
        resp = generate_with_retry(
            client=genai_client,
            model_name=gemini_model_name,
            system_prompt=primary_system_message,
            payload_obj=payload,
            retries=4,
            backoff_s=5,
        )

        result = extract_json_object(resp.text)

        validate_units_keys_only(result, payload)
        result = reorder_units_to_input_order(result, payload)

        all_out.extend(result["units"])

        print("OK")

    except Exception as e:
        print(f"FAILED | {type(e).__name__}: {e}")
        raise

df_trans = pd.DataFrame(all_out)
print("Total translated units:", len(df_trans))
df_trans.head(10)

Packet 1/5 ... retry 1/4 after transport error: RemoteProtocolError: Server disconnected without sending a response.
OK
Packet 2/5 ... OK
Packet 3/5 ... OK
Packet 4/5 ... OK
Packet 5/5 ... OK
Total translated units: 199


,unit_key,translated_text
0,dff2455410d362f483bc04508cbcb5753bca52e3,耶穌的復活是歷史上證據最充分的事件
1,28f9e896771634fe5b28da1b42499b9af0f75317,歷史
2,f68d6449d4fb305e4ee8fbc2c63e27b3c8804cbf,使徒約翰
3,a4653f3314eea5bd559a4ee5a5d4fe270c468cde,使徒馬太
4,868584564de32b98e7ea2cdfbf70b0fff67c3f03,約翰·馬可
5,bb0c7e00990a6f81b12f907151f44957605ce468,醫生路加，保羅的同伴
6,32425388bc206477680f01e421b5f721020989c8,醫生路加，保羅的同伴
7,7aef81e2f7aec81230d0a96a083f7441fb6d6ac3,使徒行傳
8,b14fd7d7025e63af23543b038ac1a4fbd28dacb0,約翰福音
9,d279e3b729c3b4b8eef8d006af038782951c6807,路加福音


### Save translated phrases to timestamped json file

In [60]:
from datetime import datetime
import re
import json

def slugify_for_filename(s: str) -> str:
    """
    Make a filesystem-friendly slug from an arbitrary label.
    - Converts any non-alphanumeric/underscore runs to a single underscore
    - Trims leading/trailing underscores
    """
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_")

def timestamp_yyyymmdd_hhmm(dt: datetime | None = None) -> str:
    dt = dt or datetime.now()
    return dt.strftime("%Y%m%d_%H%M")

ts = timestamp_yyyymmdd_hhmm()
lang_slug = slugify_for_filename(target_language)

translations_out_path = JSON_DIR / f"translations_{lang_slug}_{ts}.json"

print("lang_slug:", lang_slug)
print("timestamp:", ts)

translations_out_path.write_text(
    json.dumps(all_out, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Wrote translations:", translations_out_path.relative_to(PROJECT_ROOT))

lang_slug: Traditional_Chinese
timestamp: 20260321_2202
Wrote translations: json_files\translations_Traditional_Chinese_20260321_2202.json


## Swap out phrases and save to svg

In [69]:
# unit_key → translated_text (FROM MEMORY)
trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

print("Translations in memory:", len(trans_map))


Translations in memory: 199


In [70]:
import pandas as pd

trans_map = {r["unit_key"]: r["translated_text"] for r in all_out}

df_apply = df_units.copy()
df_apply["translated_text"] = df_apply["unit_key"].map(trans_map)

print("Rows in df_apply:", len(df_apply))
print("Translated rows:", df_apply["translated_text"].notna().sum())
print("Missing translations:", df_apply["translated_text"].isna().sum())

df_apply[["source_file", "unit_key", "unit_type", "element_path", "tspan_idx", "translated_text"]].head(10)

Rows in df_apply: 199
Translated rows: 199
Missing translations: 0


,source_file,unit_key,unit_type,element_path,tspan_idx,translated_text
0,BibleStructureTimelineNT_2024.svg,dff2455410d362f483bc04508cbcb5753bca52e3,tspan,svg/g#Headline/text[1],0,耶穌的復活是歷史上證據最充分的事件
1,BibleStructureTimelineNT_2024.svg,28f9e896771634fe5b28da1b42499b9af0f75317,tspan,svg/g#NT_History/text[1],0,歷史
2,BibleStructureTimelineNT_2024.svg,f68d6449d4fb305e4ee8fbc2c63e27b3c8804cbf,tspan,svg/g#NT_History/text[2],0,使徒約翰
3,BibleStructureTimelineNT_2024.svg,a4653f3314eea5bd559a4ee5a5d4fe270c468cde,tspan,svg/g#NT_History/text[3],0,使徒馬太
4,BibleStructureTimelineNT_2024.svg,868584564de32b98e7ea2cdfbf70b0fff67c3f03,tspan,svg/g#NT_History/text[4],0,約翰·馬可
5,BibleStructureTimelineNT_2024.svg,bb0c7e00990a6f81b12f907151f44957605ce468,tspan,svg/g#NT_History/text[5],0,醫生路加，保羅的同伴
6,BibleStructureTimelineNT_2024.svg,32425388bc206477680f01e421b5f721020989c8,tspan,svg/g#NT_History/text[6],0,醫生路加，保羅的同伴
7,BibleStructureTimelineNT_2024.svg,7aef81e2f7aec81230d0a96a083f7441fb6d6ac3,tspan,svg/g#NT_History/g#Acts_Box/text[1],1,使徒行傳
8,BibleStructureTimelineNT_2024.svg,b14fd7d7025e63af23543b038ac1a4fbd28dacb0,tspan,svg/g#NT_History/g#John_Box/text[1],1,約翰福音
9,BibleStructureTimelineNT_2024.svg,d279e3b729c3b4b8eef8d006af038782951c6807,tspan,svg/g#NT_History/g#Luke_Box/text[1],1,路加福音


In [71]:
from datetime import datetime
from pathlib import Path
from lxml import etree
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

target_language_slug = re.sub(r"[^A-Za-z0-9]+", "_", target_language).strip("_").lower()
timestamp = datetime.now().strftime("%Y%m%d_%H%M")


def localname(tag) -> str:
    if not isinstance(tag, str):
        return ""
    return tag.split("}", 1)[1] if tag.startswith("{") else tag

def find_by_element_path(root, path: str):
    """
    Resolve paths of the form:
    svg/g[1]/text[3]
    svg/g#Layer_1/text#text42
    svg/g[2]/text[1]/tspan[3]
    """
    if not path:
        return None

    parts = path.split("/")
    cur = root

    # first part should describe the root svg
    first = parts[0]
    if not first.startswith("svg"):
        return None

    for part in parts[1:]:
        m = re.fullmatch(r"([A-Za-z0-9_:-]+)(?:#(.+)|\[(\d+)\])?", part)
        if not m:
            return None

        tag, el_id, idx = m.groups()
        candidates = [
            child for child in cur
            if isinstance(getattr(child, "tag", None), str) and localname(child.tag) == tag
        ]

        if el_id is not None:
            match = None
            for child in candidates:
                if child.get("id") == el_id:
                    match = child
                    break
            if match is None:
                return None
            cur = match

        elif idx is not None:
            idx0 = int(idx) - 1  # stored paths are 1-based
            if idx0 < 0 or idx0 >= len(candidates):
                return None
            cur = candidates[idx0]

        else:
            if not candidates:
                return None
            cur = candidates[0]

    return cur


def set_text_preserve_none(el, new_text: str):
    """
    Replace text content safely.
    """
    el.text = "" if new_text is None else str(new_text)


def apply_translations_to_svg(
    svg_path: Path,
    units_for_file: pd.DataFrame,
    target_language_slug: str,
    timestamp: str,
    output_dir: Path,
) -> Path:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    updated = 0
    not_found = 0
    skipped_blank = 0

    # sort for reproducibility and to process parent text before tspans consistently
    units_for_file = units_for_file.sort_values(
        ["element_path", "unit_type", "tspan_idx"],
        na_position="last"
    )

    for _, u in units_for_file.iterrows():
        new_text = u["translated_text"]
        if pd.isna(new_text):
            skipped_blank += 1
            continue

        el = find_by_element_path(root, u["element_path"])
        if el is None:
            not_found += 1
            continue

        if u["unit_type"] == "text":
            set_text_preserve_none(el, new_text)
            updated += 1

        elif u["unit_type"] == "tspan":
            idx = u.get("tspan_idx")
            if pd.isna(idx):
                not_found += 1
                continue

            tspans = el.findall(".//svg:tspan", namespaces=NS)
            idx = int(idx)

            if idx < 0 or idx >= len(tspans):
                not_found += 1
                continue

            set_text_preserve_none(tspans[idx], new_text)
            updated += 1

    out_path = output_dir / f"{svg_path.stem}_{target_language_slug}_{timestamp}{svg_path.suffix}"
    tree.write(str(out_path), encoding="utf-8", xml_declaration=True)

    print(
        f"{svg_path.name} -> {out_path.name} | "
        f"updated={updated} | not_found={not_found} | skipped_blank={skipped_blank}"
    )
    return out_path

In [72]:
df_apply["unit_type"].value_counts(dropna=False)

unit_type
tspan    199
Name: count, dtype: int64

In [73]:
df_apply[df_apply["translated_text"].notna()][
    ["source_file", "unit_type", "element_path", "tspan_idx", "source_text", "translated_text"]
].head(20)

,source_file,unit_type,element_path,tspan_idx,source_text,translated_text
0,BibleStructureTimelineNT_2024.svg,tspan,svg/g#Headline/text[1],0,The Resurrection of Jesus is the Best Attested...,耶穌的復活是歷史上證據最充分的事件
1,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[1],0,History,歷史
2,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[2],0,John the Apostle,使徒約翰
3,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[3],0,Matthew the Apostle,使徒馬太
4,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[4],0,John Mark,約翰·馬可
5,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[5],0,"Luke the Physician, Companion of Paul",醫生路加，保羅的同伴
6,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/text[6],0,"Luke the Physician, Companion of Paul",醫生路加，保羅的同伴
7,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/g#Acts_Box/text[1],1,Acts,使徒行傳
8,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/g#John_Box/text[1],1,John,約翰福音
9,BibleStructureTimelineNT_2024.svg,tspan,svg/g#NT_History/g#Luke_Box/text[1],1,Luke,路加福音


In [74]:
out_files = []

for source_file, units_for_file in df_apply.groupby("source_file", sort=True):
    svg_path = SVG_SOURCE_DIR / source_file
    out_file = apply_translations_to_svg(
        svg_path=svg_path,
        units_for_file=units_for_file,
        target_language_slug=target_language_slug,
        timestamp=timestamp,
        output_dir=SVG_OUTPUT_DIR,
    )
    out_files.append(out_file)

print("Wrote files:", len(out_files))
for p in out_files:
    print(" -", p.relative_to(PROJECT_ROOT))

BibleStructureTimelineNT_2024.svg -> BibleStructureTimelineNT_2024_traditional_chinese_20260321_2222.svg | updated=77 | not_found=0 | skipped_blank=0
BibleStructureTimelineOT_2024.svg -> BibleStructureTimelineOT_2024_traditional_chinese_20260321_2222.svg | updated=122 | not_found=0 | skipped_blank=0
Wrote files: 2
 - svg_output_files\BibleStructureTimelineNT_2024_traditional_chinese_20260321_2222.svg
 - svg_output_files\BibleStructureTimelineOT_2024_traditional_chinese_20260321_2222.svg


### Export translation table

In [77]:
from datetime import datetime
import pandas as pd
import re

def slugify_for_filename(s: str) -> str:
    s = (s or "").strip()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
lang_slug = slugify_for_filename(target_language)

# Build export dataframe
df_export = (
    df_apply.loc[:, ["source_text", "translated_text", "group_stack"]]
    .copy()
)

print("Rows in export df:", len(df_export))
print("Missing translated_text:", df_export["translated_text"].isna().sum())

# Output paths
csv_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.csv"
xlsx_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.xlsx"
md_path = SVG_OUTPUT_DIR / f"translations_table_{lang_slug}_{timestamp}.md"

# Write files
df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_export.to_excel(xlsx_path, index=False)

md_text = df_export.to_markdown(index=False)
md_path.write_text(md_text, encoding="utf-8")

print("Wrote CSV :", csv_path.relative_to(PROJECT_ROOT))
print("Wrote XLSX:", xlsx_path.relative_to(PROJECT_ROOT))
print("Wrote MD  :", md_path.relative_to(PROJECT_ROOT))

df_export.head(10)

Rows in export df: 199
Missing translated_text: 0
Wrote CSV : svg_output_files\translations_table_traditional_chinese_20260321_2235.csv
Wrote XLSX: svg_output_files\translations_table_traditional_chinese_20260321_2235.xlsx
Wrote MD  : svg_output_files\translations_table_traditional_chinese_20260321_2235.md


,source_text,translated_text,group_stack
0,The Resurrection of Jesus is the Best Attested...,耶穌的復活是歷史上證據最充分的事件,Headline
1,History,歷史,NT_History
2,John the Apostle,使徒約翰,NT_History
3,Matthew the Apostle,使徒馬太,NT_History
4,John Mark,約翰·馬可,NT_History
5,"Luke the Physician, Companion of Paul",醫生路加，保羅的同伴,NT_History
6,"Luke the Physician, Companion of Paul",醫生路加，保羅的同伴,NT_History
7,Acts,使徒行傳,NT_History / Acts_Box
8,John,約翰福音,NT_History / John_Box
9,Luke,路加福音,NT_History / Luke_Box
